#### Day 7
  - Task Queues (Celery, Python-rq)
  - Building APIs using FastAPI
    - WSGI / ASGI frameworks overview
    - Starlette
    - Other framework (Flask, Falcon, aiohttp, ...)
  - Gradio (overview)
  - Streamlit framework (with more details)
   

In [1]:
from rq import Queue

In [2]:
from valkey import Valkey as Redis

In [3]:
queue = Queue(name="default", connection=Redis())

In [53]:
Queue?

Init signature:
Queue(
    name: str = 'default',
    connection: ForwardRef('Redis') | None = None,
    default_timeout: int | None = None,
    is_async: bool = True,
    job_class: str | type['Job'] | None = None,
    serializer: rq.serializers.Serializer | str | None = None,
    death_penalty_class: type[rq.timeouts.BaseDeathPenalty] | None = <class 'rq.timeouts.UnixSignalDeathPenalty'>,
    **kwargs,
)
Docstring:      <no docstring>
Init docstring:
Initializes a Queue object.

Args:
    name (str, optional): The queue name. Defaults to 'default'.
    default_timeout (Optional[int], optional): Queue's default timeout. Defaults to None.
    connection (Optional[Redis], optional): Redis connection. Defaults to None.
    is_async (bool, optional): Whether jobs should run "async" (using the worker).
        If `is_async` is false, jobs will run on the same process from where it was called. Defaults to True.
    job_class (Union[str, 'Job', optional): Job class or a string referencing th

In [7]:
cd task_queues/python_rq

/Users/chandra/Training/APBSA/apbsa_feb_2026/Day_7/task_queues/python_rq


In [8]:
from simple_tasks import add, add_slow

In [9]:
add(10, 20)

add method invoked: x=10, y=20


30

In [10]:
job = queue.enqueue(add, 10, 20)

In [13]:
job.get_status()

<JobStatus.FINISHED: 'finished'>

In [14]:
job.result

30

In [15]:
r1 = queue.enqueue(add_slow, 12, 13)

In [16]:
r1.get_status()

<JobStatus.STARTED: 'started'>

In [17]:
r1.get_status()

<JobStatus.FINISHED: 'finished'>

In [18]:
jobs = []

In [ ]:
from random import randint



In [34]:
x = randint(10, 50)
y = randint(10, 50)
r = queue.enqueue(add_slow, x, y)
jobs.append(r)
print("Enqueued job with arguments: ", x, y )

Enqueued job with arguments:  45 30


In [52]:
for job in jobs:
    if job.get_status() == "finished":
        print("Result: ", job.result)
    else:
        print("Job not finished yet. Status: ", job.get_status())

Result:  58
Result:  68
Result:  49
Result:  52
Result:  69
Result:  70
Result:  33
Result:  68
Result:  45
Result:  75


In [63]:
job = queue.enqueue(add_slow, 10, 20)

In [68]:
job.get_status()

<JobStatus.FINISHED: 'finished'>

In [69]:
redis = Redis()

In [70]:
redis["test"] = "Hello, Redis!"
print(redis["test"])

b'Hello, Redis!'


In [71]:
redis.set("testdata", "This is some test data.", ex=15)


True

In [78]:

print(redis.get("testdata"))
redis["testdata"]

None


KeyError: 'testdata'

In [79]:
ps = redis.pubsub()
print(ps)

In [89]:
ps.subscribe?

Signature: ps.subscribe(*args, **kwargs)
Docstring:
Subscribe to channels. Channels supplied as keyword arguments expect
a channel name as the key and a callable as the value. A channel's
callable will be invoked automatically when a message is received on
that channel rather than producing a message via ``listen()`` or
``get_message()``.
File:      /opt/anaconda3/envs/myenv/lib/python3.14/site-packages/valkey/client.py
Type:      method

In [87]:
def testfn(message):
    print("testfn(): received message: ", message)

def welcome(message):
    print("welcome(): received message: ", message)

ps.subscribe(my_channel=[testfn, welcome])


In [88]:
for ev in ps.listen():
    print("Received event: ", ev)

Received event:  {'type': 'subscribe', 'pattern': None, 'channel': b'my_channel', 'data': 1}


TypeError: 'list' object is not callable

In [82]:
redis.publish("my_channel", "Hello, Pub/Sub!")

1

In [90]:
from simple_tasks_using_celery import add, add_slow

In [100]:
r = add_slow.delay(10, 20)
r

<AsyncResult: d6db076f-dbce-40e9-b310-f6c1bfe09cf3>

In [101]:
r.status

'PENDING'

In [104]:
r.status

'SUCCESS'

In [105]:
r.result

30

#### WSGI - Web Server Gateway Interface

CGI -> Common Gateway Interface (Old-school, not scalable - one process per request)

ASGI -> Asynchronous Server Gateway Interface
  - The callout functions for requests can now be coroutines themselves
  


##### Web API (REST-like semantics):
  1. GET -> Retrieve
  2. POST -> Create
  3. PATCH -> Update
  4. PUT -> Replace
  5. DELETE -> Delete (remove)
  

In [108]:
pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]
Note: you may need to restart the kernel to use updated packages.


In [111]:
cd ../..

/Users/chandra/Training/APBSA/apbsa_feb_2026/Day_7


In [106]:
import requests

In [112]:
import dotenv
dotenv.load_dotenv("../../dotenv.sh")

True

In [ ]:
import os
os.environ["OPENWEATHER_API_KEY"]

In [117]:
owm_url = "http://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Chennai",
    "appid": os.environ["OPENWEATHER_API_KEY"],
    "units": "metric"
}

res = requests.get(owm_url, params=params)
res.status_code

200

In [118]:
res.json()

{'coord': {'lon': 80.2785, 'lat': 13.0878},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01d'}],
 'base': 'stations',
 'main': {'temp': 29.09,
  'feels_like': 29.2,
  'temp_min': 29.09,
  'temp_max': 29.09,
  'pressure': 1012,
  'humidity': 45,
  'sea_level': 1012,
  'grnd_level': 1011},
 'visibility': 10000,
 'wind': {'speed': 4.68, 'deg': 119, 'gust': 3.31},
 'clouds': {'all': 5},
 'dt': 1772520918,
 'sys': {'country': 'IN', 'sunrise': 1772499247, 'sunset': 1772542082},
 'timezone': 19800,
 'id': 1264527,
 'name': 'Chennai',
 'cod': 200}

In [119]:
res = requests.get("http://localhost:8000/hello")
res.status_code

200

In [121]:
for key, value in res.headers.items():
    print(f"{key}: {value}")

date: Tue, 03 Mar 2026 07:04:19 GMT
server: uvicorn
content-length: 27
content-type: application/json


In [122]:
res.json()

{'message': 'Hello, World!'}

In [125]:
res = requests.get("http://localhost:8000/users")
if res.ok:
    print(res.json())
else:
    print("Error: ", res.status_code)

{}


In [126]:
res = requests.post("http://localhost:8000/users", json={"name": "Alice"})

In [127]:
res.status_code

422

In [128]:
res = requests.post("http://localhost:8000/users", 
                    json={"name": "Alice", "role": "admin", "score": 95.0})
res.status_code

201

In [129]:
res = requests.post("http://localhost:8000/users", 
                    json={"name": "Sam", "role": "developer", "score": 65.0})
res.status_code

201

In [130]:
res = requests.post("http://localhost:8000/users", 
                    json={"name": "Jim", "role": "admin", "score": 65.0})
res.status_code

201

In [131]:
res = requests.get("http://localhost:8000/users")
if res.ok:
    print(res.json())
else:
    print("Error: ", res.status_code)

{'Alice': {'name': 'Alice', 'role': 'admin', 'score': 95.0}, 'Sam': {'name': 'Sam', 'role': 'developer', 'score': 65.0}, 'Jim': {'name': 'Jim', 'role': 'admin', 'score': 65.0}}


In [132]:
res = requests.get("http://localhost:8000/users/Jim")
if res.ok:
    print(res.json())
else:    
    print("Error: ", res.status_code)

{'name': 'Jim', 'role': 'admin', 'score': 65.0}
